# 06 — Grover Search & Amplitude Amplification

Grover's search algorithm finds a marked item in an unstructured database of size $N = 2^n$ in $O(\sqrt{N})$ oracle queries, compared to the classical average of $N/2$ queries.

In this notebook, we follow the amplitude amplification process step by step for $n = 2$ qubits ($N = 4$) with marked item $|\omega\rangle = |11\rangle$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from quantum_ed.states import ket0, normalize
from quantum_ed.gates import H, kron_n, apply
from quantum_ed.linalg import dagger, projector

## 1. Initial State: Uniform Superposition $|s\rangle$

We start in $|00\rangle$ and apply Hadamard gates to all qubits:

$$|s\rangle = H^{\otimes 2}|00\rangle = \frac{1}{2}(|00\rangle + |01\rangle + |10\rangle + |11\rangle)$$

In [ ]:
state_00 = kron_n(ket0(), ket0())
H2 = kron_n(H, H)
s = apply(H2, state_00)

labels = ['00', '01', '10', '11']
print("Initial uniform state amplitudes:")
for label, amp in zip(labels, s.flatten()):
    print(f"  |{label}>: {amp.real:+.3f}")

## 2. The Phase Oracle $U_\omega$

The oracle recognizes the target state $|\omega\rangle = |11\rangle$ and flips its phase:

$$U_\omega = I - 2|\omega\rangle\langle\omega|$$

For $|11\rangle$, the amplitude changes from $+1/2$ to $-1/2$, while all other amplitudes remain $+1/2$.

In [ ]:
target = np.array([[0], [0], [0], [1]], dtype=complex)  # |11>
I4 = np.eye(4, dtype=complex)
U_omega = I4 - 2.0 * projector(target)

after_oracle = apply(U_omega, s)

print("Amplitudes after oracle U_omega:")
for label, amp in zip(labels, after_oracle.flatten()):
    print(f"  |{label}>: {amp.real:+.3f}")

## 3. The Diffusion Operator $D$

The Grover diffusion operator reflects amplitudes about their mean $\mu$:

$$D = 2|s\rangle\langle s| - I$$

Mean amplitude after oracle: $\mu = \frac{1}{4}(1/2 + 1/2 + 1/2 - 1/2) = 1/4$.
- Unmarked states: $2\mu - 1/2 = 2(1/4) - 1/2 = 0$.
- Marked state $|11\rangle$: $2\mu - (-1/2) = 2(1/4) + 1/2 = 1$.

In [ ]:
D = 2.0 * projector(s) - I4
after_diffusion = apply(D, after_oracle)

print("Amplitudes after diffusion D:")
for label, amp in zip(labels, after_diffusion.flatten()):
    print(f"  |{label}>: {amp.real:+.3f}")

probabilities = np.abs(after_diffusion.flatten()) ** 2
print("\nFinal measurement probabilities:")
for label, p in zip(labels, probabilities):
    print(f"  |{label}>: {p:.4f}")

assert np.isclose(probabilities[3], 1.0)

## 4. Geometric Interpretation in the 2D Subspace

The entire Grover evolution occurs in a 2D plane spanned by:
1. The target state $|\omega\rangle = |11\rangle$.
2. The superposition of non-target states $|\omega^\perp\rangle = \frac{1}{\sqrt{3}}(|00\rangle + |01\rangle + |10\rangle)$.

The initial state is:
$$|s\rangle = \cos(\theta/2)|\omega^\perp\rangle + \sin(\theta/2)|\omega\rangle,$$
where $\sin(\theta/2) = 1/\sqrt{N} = 1/2 \implies \theta/2 = 30^\circ$.

Each Grover iteration $G = D U_\omega$ rotates the state by an angle $\theta = 60^\circ$ towards $|\omega\rangle$.
After exactly $R = 1$ iteration, the state reaches $30^\circ + 60^\circ = 90^\circ$, which is perfectly aligned with $|\omega\rangle$!

In [ ]:
omega_perp = np.array([[1], [1], [1], [0]], dtype=complex) / np.sqrt(3)

# Coordinates in (|omega_perp>, |omega>) basis
def to_2d(ket):
    x = np.real((dagger(omega_perp) @ ket).item())
    y = np.real((dagger(target) @ ket).item())
    return x, y

x_s, y_s = to_2d(s)
x_oracle, y_oracle = to_2d(after_oracle)
x_diff, y_diff = to_2d(after_diffusion)

print(f"Initial state |s>:       (x={x_s:.3f}, y={y_s:.3f}) -> angle = {np.degrees(np.arctan2(y_s, x_s)):.1f} deg")
print(f"After oracle U_omega:   (x={x_oracle:.3f}, y={y_oracle:.3f}) -> angle = {np.degrees(np.arctan2(y_oracle, x_oracle)):.1f} deg")
print(f"After diffusion D:      (x={x_diff:.3f}, y={y_diff:.3f}) -> angle = {np.degrees(np.arctan2(y_diff, x_diff)):.1f} deg")

## Exercises

1) What happens if you apply a second Grover iteration ($R = 2$)? Check if the success probability increases or decreases (phenomenon of **over-rotation**).
2) Generalize to $n = 3$ qubits ($N = 8$). Calculate the optimal number of iterations $R \approx \frac{\pi}{4}\sqrt{N} \approx 2.22 \implies 2$ iterations, and verify the final success probability.